# This file is for experimenting and seeing what works and how. All the final code will be available in main.py.
- This was my first attempt.
- I got stuck at the email parsing phase.
- I did not realise there were libraries to help. Attempt #2 available in ex-classifier.ipynb

In [16]:
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
import os
import pickle

In [17]:
scope = ["https://www.googleapis.com/auth/gmail.modify"]

In [18]:
creds = None

In [19]:
if os.path.exists("token.pickle"):
    with open("token.pickle", "rb") as token:
        creds = pickle.load(token)

In [20]:
if not creds:
    flow = InstalledAppFlow.from_client_secrets_file(
        "credentials.json",
        scope
    )

    creds = flow.run_local_server(port=0)

    with open("token.pickle", "wb") as token:
        pickle.dump(creds, token)

In [21]:
service = build("gmail", "v1", credentials=creds)

In [22]:
from pprint import pprint
results = service.users().messages().list(
    userId="me"
).execute()

pprint(f"first 10 msg ids: {results["messages"][:10]}\nnext page token: {results["nextPageToken"]}\nresult size estimate: {"resultSizeEstimate"}")

("first 10 msg ids: [{'id': '19e9dd2429d2da77', 'threadId': "
 "'19e9dd2429d2da77'}, {'id': '19e99542404b66d0', 'threadId': "
 "'19e99542404b66d0'}, {'id': '19e98d4b48ef2d88', 'threadId': "
 "'19e98d4b48ef2d88'}, {'id': '19e98926ef327e6c', 'threadId': "
 "'19e9873be4068fbe'}, {'id': '19e9873be4068fbe', 'threadId': "
 "'19e9873be4068fbe'}, {'id': '19e9852a2306c1a5', 'threadId': "
 "'19e9852a2306c1a5'}, {'id': '19e9819b3f00b6f5', 'threadId': "
 "'19e9819b3f00b6f5'}, {'id': '19e97c6ab230bb6a', 'threadId': "
 "'19e97c6ab230bb6a'}, {'id': '19e971fbaaf2c8e0', 'threadId': "
 "'19e971fbaaf2c8e0'}, {'id': '19e95aaae9a419ad', 'threadId': "
 "'19e95aaae9a419ad'}]\n"
 'next page token: 03672064951464537470\n'
 'result size estimate: resultSizeEstimate')


In [23]:
profile = service.users().getProfile(userId="me").execute()

print(profile)

{'emailAddress': 'syedmrihaan@gmail.com', 'messagesTotal': 1216, 'threadsTotal': 1119, 'historyId': '600059'}


In [24]:
results = service.users().messages().list(
    userId="me",
    labelIds=["INBOX"]
).execute()

print(results["resultSizeEstimate"])

201


# **Our OAuth has been implemented.**

### **Next Steps:**
***Data Collection***
- Collect message ID 
- Collect sender name 
- Collect sender domain, sender local part 
- Collect combined text(subject + body) 
- Collect word count of body 
- Collect number of numerical characters(digits) in body. 
- Collect link count(useful for promotional mails or maybe school mails? idk) 
- Collect currency count(useful for promotional i think) 
- Collect presence of otp-like numbers 
- Collect exclamation mark count(spammy emails) 

***Final Dataframe***
- Sender info 
- Combined text
- Word count
- Digit count
- Mail folder
- Message ID(DO NOT TRAIN ON THIS)

***Marking Targets***
- Perform unsupervised KMeans to get clusters.

***Model and Final pipeline***
- Gaussian naive bayes.

In [25]:
# get msg IDs which we will work with

ids = service.users().messages().list(
    userId = "me",
    q = "newer_than:90d",
    maxResults = 500
).execute()

In [26]:
ids["messages"][:10]

[{'id': '19e9dd2429d2da77', 'threadId': '19e9dd2429d2da77'},
 {'id': '19e99542404b66d0', 'threadId': '19e99542404b66d0'},
 {'id': '19e98d4b48ef2d88', 'threadId': '19e98d4b48ef2d88'},
 {'id': '19e98926ef327e6c', 'threadId': '19e9873be4068fbe'},
 {'id': '19e9873be4068fbe', 'threadId': '19e9873be4068fbe'},
 {'id': '19e9852a2306c1a5', 'threadId': '19e9852a2306c1a5'},
 {'id': '19e9819b3f00b6f5', 'threadId': '19e9819b3f00b6f5'},
 {'id': '19e97c6ab230bb6a', 'threadId': '19e97c6ab230bb6a'},
 {'id': '19e971fbaaf2c8e0', 'threadId': '19e971fbaaf2c8e0'},
 {'id': '19e95aaae9a419ad', 'threadId': '19e95aaae9a419ad'}]

In [27]:
ids = ids["messages"]

In [28]:
len(ids)

288

In [29]:
m_id = ids[5]["id"]

In [30]:
msg = service.users().messages().get(
    userId = "me",
    id = m_id
).execute()

In [51]:
msg # plug into json crack to visualize data

{'id': '19e9852a2306c1a5',
 'threadId': '19e9852a2306c1a5',
 'labelIds': ['INBOX'],
 'snippet': 'Syed Muhammad Rihaan Oakridge International School, Gachibowli Dear Syed Muhammad Rihaan It is with great pleasure that we inform you of your selection for the AI Internship for Young Achievers (AIYA)',
 'payload': {'partId': '',
  'mimeType': 'multipart/alternative',
  'filename': '',
  'headers': [{'name': 'Delivered-To', 'value': 'syedmrihaan@gmail.com'},
   {'name': 'Received',
    'value': 'by 2002:a05:7118:82:b0:1d9:22c8:9c5c with SMTP id 2csp313227gge;        Fri, 5 Jun 2026 08:06:53 -0700 (PDT)'},
   {'name': 'X-Received',
    'value': 'by 2002:a17:90b:1b50:b0:36b:a162:a1be with SMTP id 98e67ed59e1d1-3713049177fmr2577187a91.4.1780672012957;        Fri, 05 Jun 2026 08:06:52 -0700 (PDT)'},
   {'name': 'ARC-Seal',
    'value': 'i=1; a=rsa-sha256; t=1780672012; cv=none;        d=google.com; s=arc-20240605;        b=TWN3Wz1cmzbtiU04aKDMc49E0NyCUfRzpXvZ0AIudvou7XBueBN7tDnWrntFga5o8d      

In [32]:
def get_header(msg, name):
    for h in msg["payload"]["headers"]:
        if h["name"] == name:
            return h["value"]
    return None

# **Main Challenge**
Some mails are in plain text, while some are in the form of html, base-64 encoded, and the important information(body) resides in different parts of the JSON the gmail API sends back. We need to figure out all the possible paths where this body can be, and parse each mail on the basis of that. 

In [33]:
# get all messages via batch object

messages = [] # empty list for all messages

def callback(request_id, response, exception): # api returns internal request ID, result of our request, and exception(if encountered)
    if exception is None: 
        messages.append(response)
    else:
        print("-" * 50) 
        print("oopsies, we got an error :<. the error is: {}".format(exception))

batch_size = 20 # since google api limits batch sizes to 100, but recommends <50

for start in range(0, len(ids), batch_size):

    batch = service.new_batch_http_request()

    chunk = ids[start:start + batch_size]

    for mail in chunk:
        batch.add(
            service.users().messages().get(
                userId = "me",
                id = mail["id"]
            ),
            callback = callback
        )
    batch.execute()

In [52]:
messages[1]

{'id': '19e99542404b66d0',
 'threadId': '19e99542404b66d0',
 'labelIds': ['INBOX'],
 'snippet': 'Your documents have been approved — you&#39;re all set! ͏\u200c ͏\u200c ͏\u200c ͏\u200c ͏\u200c ͏\u200c ͏\u200c ͏\u200c ͏\u200c ͏\u200c ͏\u200c ͏\u200c ͏\u200c ͏\u200c ͏\u200c ͏\u200c Hack Club Auth Hey Muhammad! Your documents have been approved — you&#39;re all set! Best, the',
 'payload': {'partId': '',
  'mimeType': 'multipart/alternative',
  'filename': '',
  'headers': [{'name': 'Delivered-To', 'value': 'syedmrihaan@gmail.com'},
   {'name': 'Received',
    'value': 'by 2002:a05:7118:82:b0:1d9:22c8:9c5c with SMTP id 2csp529164gge;        Fri, 5 Jun 2026 12:48:09 -0700 (PDT)'},
   {'name': 'X-Received',
    'value': 'by 2002:a05:7022:6b99:b0:137:e532:f53f with SMTP id a92af1059eb24-1380672bf32mr2590788c88.34.1780688889355;        Fri, 05 Jun 2026 12:48:09 -0700 (PDT)'},
   {'name': 'ARC-Seal',
    'value': 'i=1; a=rsa-sha256; t=1780688889; cv=none;        d=google.com; s=arc-20240605;  

In [35]:
len(messages) # verifying that list structure is ok and all mails have been retrieved

288

In [36]:
mime_types = set()

for message in messages:
    mime_types.add(message["payload"]["mimeType"])

In [37]:
mime_types

{'multipart/alternative',
 'multipart/mixed',
 'multipart/related',
 'text/html',
 'text/plain'}

In [38]:
mtype_msgs = {mime: [] for mime in mime_types} # dict to classify msgs based on mime type(since diff mime types have diff extraction methods)


In [53]:
for message in messages:
    mime_type = message["payload"]["mimeType"]
    mtype_msgs[mime_type].append(message)
    # classify on basis of mimetypes

In [54]:
len(mtype_msgs['multipart/related'])

2

# **First: Multipart/alternative**

In [55]:
# inspecting this form of mail on json crack, it seems that the plain text resides in [payload]][parts] where there are two parts : plain text and html.
# ill also put a counter here to check if ALL multipart/alternative have this "text/plain" part or if we'll ahve to use beautiful soup for some

counter = 0

for mail in mtype_msgs["multipart/alternative"]:
    parts = mail["payload"]["parts"]
    for part in parts:
        if part["mimeType"] == "text/plain":
            counter += 1
print(counter == len(mtype_msgs["multipart/alternative"]))

True


In [43]:
# we see that each part has a "body" key, which itself is a dictionary consisting of the base-64 encoded version of data we want.
# also, our counter test confirms that each mail in this mime type has the plain text part.

import base64

mp_alt_body = []

for mail in mtype_msgs["multipart/alternative"]:
    parts = mail["payload"]["parts"]
    for part in parts:
        if part["mimeType"] == "text/plain":
            body = part["body"]["data"]
            text = base64.urlsafe_b64decode(body).decode("utf-8")
            mp_alt_body.append(text)



In [ ]:
mp_alt_body

# **Now we need to clean the text.**
- Bro, I'm going to actually crash out.

- Anyway, here's what we're going to do:
- Use html.unescape to get rid of special html characters(like &amp becomes ampersand: &)
- Get rid of annoying characters like /u200c(zero width non joiner, dont ask me what that means), \n(new line), \r(raw text), etc...
- Get rid of white space via (...).split and " ".join(...).

In [45]:
import html, re
def cleaner(text):
    cleaned = html.unescape(text)
    cleaned = re.sub(r'[\u200c\u200b\u200d\ufeff\xad]', "", cleaned) 
    # [] => match any one char, an since every char is unicode, regex treats \u200c(for eg) as a single char
    cleaned = " ".join(cleaned.split())
    return cleaned

### **Well that was surprisingly easy.**
# **Now lets build our records yayyaaya! (why do i do this to myself)**

In [ ]:
# now we build proper records for all these mails.
"""
we need:

- Collect message ID ✅
- Collect sender name ✅
- Collect sender domain, sender local part ✅
- Collect combined text(subject + body) ✅
- Collect word count of body ✅
- Collect number of numerical characters(digits) in body. ✅
- Collect link count(useful for promotional mails or maybe school mails? idk) ✅
- Collect currency count(useful for promotional i think) ✅
- Collect presence of otp-like numbers ✅
- Collect exclamation mark count(spammy emails) ✅

"""

def from_header(attr):
    return [headers[i]["value"] for i in range(0, len(headers)) if headers[i]["name"] == attr][0] # search headers until we get a particular value

from email.utils import parseaddr

records = []

for message in mtype_msgs["multipart/alternative"]:
    m_id = message["id"]
    payload = message["payload"]
    headers = payload["headers"]

    # mail stuff
    disp_name = from_header("From") # search headers until we find dict of "from"
    sender_name, sender_addr = parseaddr(disp_name) # parse into name and email addr
    sender_local_part, sender_domain = sender_addr.rsplit("@", 1) # parse email addr into domain and local part

    # subject
    subject = from_header("Subject")

    # body
    parts = payload["parts"]
    body = ""
    for part in parts:
        if part["mimeType"] == "text/plain":
            text = part["body"]["data"]
            text = base64.urlsafe_b64decode(text).decode("utf-8")
            text = cleaner(text)
            body += text #incase there are multiple plain/texts.

    # combined text
    combined_text = f"{subject} {body}"

    # word count
    words = combined_text.split()
    word_count = len(words)

    # digit count
    digit_count = sum(c.isdigit() for c in body)

    # exclamation mark count
    exc_mark_count = body.count("!")

    # link count
    link_count = len(re.findall(r"https?://", body)) # ? next to s means s need not be present. so https and http both work fine.

    # currency count
    currency_count = (
        body.count("$") +
        body.count("₹")
    )

    # otp-like numbers
    otp_num = len(re.findall(r'\b\d{4,8}\b', body)) # \b -> word boundary, \d: check for these many digits, {4, 8} -> need to be 4-8 digits consecutively.

    # final record
    record = [m_id, 
              sender_name, 
              sender_local_part, 
              sender_domain, 
              combined_text, 
              word_count, 
              digit_count, 
              exc_mark_count, 
              link_count, 
              currency_count, 
              otp_num]
    
    records.append(record)

In [47]:
# convert into numpy array because yes i love numpy and complicated

import numpy as np

records = np.array(records)

In [57]:
records[:5, 4]

array(["[Hack Club] Your identity verification has been approved! Hey Muhammad! Your documents have been approved — you're all set! Best, the Hack Club team -- Hack Club is an open source and financially transparent 501(c)(3) nonprofit. This is yours. https://hackclub.com",
       "Happy World Environment Day 2026! *Dear Greenies,* *Greetings from RUR GreenLife! 🌱* Happy World Environment Day! Thank you for being part of the green movement and supporting a more sustainable future. Every small action towards reducing waste, conserving resources, and caring for our environment creates a lasting impact. Together, let's continue building a cleaner, greener world for generations to come. <https://drive.google.com/file/d/1rXs8dNxANlmjPK1tbCNz6UME4OlpuCpM/view?usp=drive_link&ct=t(EMAIL_CAMPAIGN_8_21_2024_17_30_COPY_01)&mc_cid=e6847227db&mc_eid=UNIQID> <https://www.instagram.com/reel/DZEk33SBCTx/?ct=t(EMAIL_CAMPAIGN_8_21_2024_17_30_COPY_01)&mc_cid=e6847227db&mc_eid=UNIQID> <https://youtu.be/jm

### **LETS GOOOO ITS WORKING YESSIRRRR I LOVE YALL**

# **Let's build the dataframe now :>**

In [49]:
import pandas as pd

df = pd.DataFrame(
    data = records,
    columns = ["message_id", "sender_name", "sender_local_part", "sender_doman", "combined_text", "word_count", "digit_count", "exc_mark_count"
               ,"link_count", "currency_count", "otp_count"]
).set_index("message_id")

In [50]:
df

,sender_name,sender_local_part,sender_doman,combined_text,word_count,digit_count,exc_mark_count,link_count,currency_count,otp_count
message_id,,,,,,,,,,
19e99542404b66d0,Hack Club,identity,hackclub.com,[Hack Club] Your identity verification has bee...,40,4,2,1,0,0
19e98d4b48ef2d88,RUR Greenlife,info,rur.co.in,Happy World Environment Day 2026! *Dear Greeni...,123,108,2,8,0,0
19e98926ef327e6c,Muhammad Rihaan,syedmrihaan,gmail.com,Fwd: Revised Offer Letter for AIYA and Confirm...,1185,190,0,3,0,14
19e9873be4068fbe,Katherine Yang,admissions,corporategurukul.com,Revised Offer Letter for AIYA and Confirmation...,1114,87,0,0,0,10
19e9852a2306c1a5,Katherine Yang,admissions,corporategurukul.com,Syed Muhammad Rihaan | Offer Letter for AI In...,973,83,0,0,0,10
...,...,...,...,...,...,...,...,...,...,...
19cdd954ae14af03,GeeksforGeeks,no-reply,geeksforgeeks.org,"Meet India's Best GATE Faculty 🇮🇳 Hi Geeks, If...",189,14,1,0,0,3
19cdc314ef20e559,Apple,News,insideapple.apple.com,iPhone 17e is here. Upgrade the easy way. At A...,920,1838,0,25,4,9
19cd7a576d563e3d,Jake@TA,support,teachersassistantai.com.au,You can now build your own AI Templates You ca...,556,461,1,13,0,2


### ***The next day.***

I just checked the code and I noticed I already calculated some stuff but didnt implement them in the code for retrieving records for multipart/alternative. Just wanted to clarify that that's fine, as this is not the final notebook. It's just to help me reach the code i need, since doing this in a python file would mean having to repeatedly run a file when im searching for JSON values. Final code will be in main.py, and ill probably just add sort of checkpoints throughout the notebook to get a look at what i've really done, and what remains.

### ***Continuing the code.***
- Small problem. I'm only extracting plain text. Not... html... so tones of data loss. :/
- Let's fix that.

*Probably a good time to mention ive never used beautiful soup.*

In [60]:
# Html content resides in the same place as text/plain.
from bs4 import BeautifulSoup

# first i just wanna see how the html looks. so we'll inspect the first mail as an example.

mail = mtype_msgs["multipart/alternative"][1]
parts = mail["payload"]["parts"] # access "parts" key
for part in parts: # iterate over each part
    if part["mimeType"] == "text/html": # if the part's mimetype is text/html
        body = part["body"]["data"] 
        print(body)

PGRpdiBkaXI9Imx0ciI-PGRpdj48dGFibGUgaWQ9Im1fNTA0OTMxNDIyNzQyODQ4ODU4MnJvb3QiIGJvcmRlcj0iMCIgY2VsbHBhZGRpbmc9IjAiIGNlbGxzcGFjaW5nPSIwIiB3aWR0aD0iMTAwJSIgcm9sZT0icHJlc2VudGF0aW9uIiBzdHlsZT0iYm9yZGVyLWNvbGxhcHNlOmNvbGxhcHNlO2JhY2tncm91bmQtY29sb3I6cmdiKDI0NCwyNDQsMjQ0KSI-PHRib2R5Pjx0cj48dGQgdmFsaWduPSJ0b3AiIGFsaWduPSJjZW50ZXIiIHN0eWxlPSJ3b3JkLWJyZWFrOmJyZWFrLXdvcmQ7YmFja2dyb3VuZC1jb2xvcjp0cmFuc3BhcmVudCI-PHRhYmxlIGJvcmRlcj0iMCIgY2VsbHBhZGRpbmc9IjAiIGNlbGxzcGFjaW5nPSIwIiB3aWR0aD0iMTAwJSIgcm9sZT0icHJlc2VudGF0aW9uIiBzdHlsZT0iYm9yZGVyLWNvbGxhcHNlOmNvbGxhcHNlO21heC13aWR0aDo2NjBweCI-PHRib2R5Pjx0cj48dGQgdmFsaWduPSJ0b3AiIHN0eWxlPSJ3b3JkLWJyZWFrOmJyZWFrLXdvcmQ7YmFja2dyb3VuZC1jb2xvcjpyZ2IoMjU1LDI1NSwyNTUpIj48dGFibGUgYWxpZ249ImNlbnRlciIgYm9yZGVyPSIwIiBjZWxscGFkZGluZz0iMCIgY2VsbHNwYWNpbmc9IjAiIHdpZHRoPSIxMDAlIiByb2xlPSJwcmVzZW50YXRpb24iIHN0eWxlPSJib3JkZXItY29sbGFwc2U6Y29sbGFwc2UiPjx0Ym9keT48dHI-PHRkIHZhbGlnbj0idG9wIiBzdHlsZT0id29yZC1icmVhazpicmVhay13b3JkO2JhY2tncm91bmQtcG9zaXRpb246Y2VudGVyIGNlbnRlcjti

In [62]:
# since its base-64 encoded, we'll decode it.

decoded_body = base64.urlsafe_b64decode(body).decode("utf-8")
decoded_body

'<div dir="ltr"><div><table id="m_5049314227428488582root" border="0" cellpadding="0" cellspacing="0" width="100%" role="presentation" style="border-collapse:collapse;background-color:rgb(244,244,244)"><tbody><tr><td valign="top" align="center" style="word-break:break-word;background-color:transparent"><table border="0" cellpadding="0" cellspacing="0" width="100%" role="presentation" style="border-collapse:collapse;max-width:660px"><tbody><tr><td valign="top" style="word-break:break-word;background-color:rgb(255,255,255)"><table align="center" border="0" cellpadding="0" cellspacing="0" width="100%" role="presentation" style="border-collapse:collapse"><tbody><tr><td valign="top" style="word-break:break-word;background-position:center center;background-repeat:no-repeat;background-size:cover"><table border="0" cellpadding="0" cellspacing="0" width="100%" role="presentation" style="border-collapse:collapse"><tbody><tr><td valign="top" class="m_5049314227428488582mceColumn" id="m_5049314227

In [66]:
# now we'll use beautiful soup to parse

soup = BeautifulSoup(decoded_body, "html.parser") # (content to be parsed, parser to use)

In [71]:
text = soup.get_text(strip = True, separator = " ")
text

"Dear Greenies, Greetings from RUR GreenLife!\xa0🌱 Happy World Environment Day! Thank you for being part of the green movement and supporting a more sustainable future. Every small action towards reducing waste, conserving resources, and caring for our environment creates a lasting impact. Together, let's continue building a cleaner, greener world for generations to come. -- Regards, rur.co.in | Facebook | Twitter | +91 9326093543 RUR is proud to win recognition on international platform by SWANA ( Solid Waste Management association North America) for sustainable waste management. To know more click here: SWANA Excellence Awards To unsubscribe from this group and stop receiving emails from it, send an email to rur-green-champions+unsubscribe@rur.co.in ."

In [73]:
# i wanna see if this is different from the plain text

mail = mtype_msgs["multipart/alternative"][1]
parts = mail["payload"]["parts"] # access "parts" key
for part in parts: # iterate over each part
    if part["mimeType"] == "text/plain": # if the part's mimetype is text/html
        body = part["body"]["data"] 
        decoded_body = base64.urlsafe_b64decode(body).decode("utf-8")
decoded_body

"*Dear Greenies,*\r\n\r\n\r\n\r\n*Greetings from RUR GreenLife! 🌱*\r\n\r\nHappy World Environment Day! Thank you for being part of the green movement\r\nand supporting a more sustainable future. Every small action towards\r\nreducing waste, conserving resources, and caring for our environment\r\ncreates a lasting impact. Together, let's continue building a cleaner,\r\ngreener world for generations to come.\r\n\r\n\r\n\r\n\r\n<https://drive.google.com/file/d/1rXs8dNxANlmjPK1tbCNz6UME4OlpuCpM/view?usp=drive_link&ct=t(EMAIL_CAMPAIGN_8_21_2024_17_30_COPY_01)&mc_cid=e6847227db&mc_eid=UNIQID>\r\n<https://www.instagram.com/reel/DZEk33SBCTx/?ct=t(EMAIL_CAMPAIGN_8_21_2024_17_30_COPY_01)&mc_cid=e6847227db&mc_eid=UNIQID>\r\n<https://youtu.be/jm1SyS5Mvdg?si=bVeqbpe66ejIAcc6&ct=t(EMAIL_CAMPAIGN_8_21_2024_17_30_COPY_01)&mc_cid=e6847227db&mc_eid=UNIQID>\r\n<http://rur.co.in/?ct=t(EMAIL_CAMPAIGN_8_21_2024_17_30_COPY_01)&mc_cid=e6847227db&mc_eid=UNIQID>\r\n\r\n-- \r\n*Regards,*\r\n\r\n*rur.co.in <http:

In [ ]:
# same content?
cleaner(decoded_body)

"*Dear Greenies,* *Greetings from RUR GreenLife! 🌱* Happy World Environment Day! Thank you for being part of the green movement and supporting a more sustainable future. Every small action towards reducing waste, conserving resources, and caring for our environment creates a lasting impact. Together, let's continue building a cleaner, greener world for generations to come. <https://drive.google.com/file/d/1rXs8dNxANlmjPK1tbCNz6UME4OlpuCpM/view?usp=drive_link&ct=t(EMAIL_CAMPAIGN_8_21_2024_17_30_COPY_01)&mc_cid=e6847227db&mc_eid=UNIQID> <https://www.instagram.com/reel/DZEk33SBCTx/?ct=t(EMAIL_CAMPAIGN_8_21_2024_17_30_COPY_01)&mc_cid=e6847227db&mc_eid=UNIQID> <https://youtu.be/jm1SyS5Mvdg?si=bVeqbpe66ejIAcc6&ct=t(EMAIL_CAMPAIGN_8_21_2024_17_30_COPY_01)&mc_cid=e6847227db&mc_eid=UNIQID> <http://rur.co.in/?ct=t(EMAIL_CAMPAIGN_8_21_2024_17_30_COPY_01)&mc_cid=e6847227db&mc_eid=UNIQID> -- *Regards,* *rur.co.in <http://www.rur.co.in/> |Facebook <https://www.facebook.com/RUR.AreYouReducingReusingR

In [76]:
cleaner(text)

"Dear Greenies, Greetings from RUR GreenLife! 🌱 Happy World Environment Day! Thank you for being part of the green movement and supporting a more sustainable future. Every small action towards reducing waste, conserving resources, and caring for our environment creates a lasting impact. Together, let's continue building a cleaner, greener world for generations to come. -- Regards, rur.co.in | Facebook | Twitter | +91 9326093543 RUR is proud to win recognition on international platform by SWANA ( Solid Waste Management association North America) for sustainable waste management. To know more click here: SWANA Excellence Awards To unsubscribe from this group and stop receiving emails from it, send an email to rur-green-champions+unsubscribe@rur.co.in ."

In [108]:
# seems that plain text is more poluted with links since we're not excluding it. i wonder if this applies to all mails.
# we'll use difflib's sequence matcher.
# i also want to see how many mails have both, or only one of two

from difflib import SequenceMatcher

ratios = []
has_both = 0
only_plain = 0
only_html = 0

for mail in mtype_msgs["multipart/alternative"]:
    parts = mail["payload"]["parts"] # access "parts" key

    plain_text = None
    html_text = None

    for part in parts: # iterate over each part

        body = part["body"]["data"] 
        decoded_body = base64.urlsafe_b64decode(body).decode("utf-8") # decode

        if part["mimeType"] == "text/html":
            html_text = cleaner(BeautifulSoup(decoded_body, "html.parser").get_text(strip=True, separator=" ")) # parse
        if part["mimeType"] == "text/plain":
            plain_text = cleaner(decoded_body) # clean

    if html_text and plain_text:
        ratios.append(SequenceMatcher( None, plain_text,html_text).ratio())
        has_both += 1
    elif html_text and not plain_text:
        only_html += 1
    elif plain_text and not html_text:
        only_plain += 1


In [109]:
import numpy as np
print(f"""{np.mean(ratios)=},
{np.min(ratios)=},
{np.max(ratios)=},
{np.median(ratios)=}""")

np.mean(ratios)=np.float64(0.7561480808670181),
np.min(ratios)=np.float64(0.11133200795228629),
np.max(ratios)=np.float64(1.0),
np.median(ratios)=np.float64(0.823239328093697)


In [110]:
has_both, only_html, only_plain

(426, 2, 2)

In [111]:
# so most mails support html with very little difference. i wonder if presence of links is causing our mean and median to go down. 
# lets clean up and rmeove mails as well before checking.

from difflib import SequenceMatcher

ratios = []
has_both = 0
only_plain = 0
only_html = 0

for mail in mtype_msgs["multipart/alternative"]:
    parts = mail["payload"]["parts"] # access "parts" key

    plain_text = None
    html_text = None

    for part in parts: # iterate over each part

        body = part["body"]["data"] 
        decoded_body = base64.urlsafe_b64decode(body).decode("utf-8") # decode

        if part["mimeType"] == "text/html":
            html_text = cleaner(BeautifulSoup(decoded_body, "html.parser").get_text(strip=True, separator=" ")) # parse
        if part["mimeType"] == "text/plain":
            plain_text = re.sub(r"https?://\S+", "", cleaner(decoded_body)) # clean

    if html_text and plain_text:
        ratios.append(SequenceMatcher( None, plain_text,html_text).ratio())
        has_both += 1
    elif html_text and not plain_text:
        only_html += 1
    elif plain_text and not html_text:
        only_plain += 1


In [112]:
import numpy as np
print(f"""{np.mean(ratios)=},
{np.min(ratios)=},
{np.max(ratios)=},
{np.median(ratios)=}""")

np.mean(ratios)=np.float64(0.885404274025179),
np.min(ratios)=np.float64(0.11133200795228629),
np.max(ratios)=np.float64(1.0),
np.median(ratios)=np.float64(0.9531645569620253)


# **New plan:**

- Use html if available, else fallback to plain.
- Also record contains_html, and contains_plain.
- Also record presence of html structures like image_count, button_count, etc...
- That's it man what else did you expect?

In [ ]:
# now we build proper records for all these mails.
"""
we need:

- Collect message ID ✅
- Collect sender name ✅
- Collect sender domain, sender local part ✅
- Collect combined text(subject + body) ✅
- Collect word count of body ✅
- Collect number of numerical characters(digits) in body. ✅
- Collect link count(useful for promotional mails or maybe school mails? idk) ✅
- Collect currency count(useful for promotional i think) ✅
- Collect presence of otp-like numbers ✅
- Collect exclamation mark count(spammy emails) ✅
- Contains HTML ✅
- Contains plain text ✅
- Image count
- Button count

"""

def from_header(attr):
    return [headers[i]["value"] for i in range(0, len(headers)) if headers[i]["name"] == attr][0] # search headers until we get a particular value

from email.utils import parseaddr

records = []

for message in mtype_msgs["multipart/alternative"]:
    m_id = message["id"]
    payload = message["payload"]
    headers = payload["headers"]

    # mail stuff
    disp_name = from_header("From") # search headers until we find dict of "from"
    sender_name, sender_addr = parseaddr(disp_name) # parse into name and email addr
    sender_local_part, sender_domain = sender_addr.rsplit("@", 1) # parse email addr into domain and local part

    # subject
    subject = from_header("Subject")

    # body
    parts = payload["parts"]
    body = ""
    contains_html = sum([part["mimeType"] == "text/html" for part in parts]) # iterate over each part and check if its text/html. 
    contains_plain = sum([part["mimeType"] == "text/plain" for part in parts]) # iterate over each part and check if its text/plain. 
    # Since true vals are 1, sum should return 1 if there is atleast 1 text/html part

    for part in parts:

        if contains_html and part["mimeType"] == "text/plain": continue # we dont care about plain when html is present

        text = part["body"]["data"]
        text = base64.urlsafe_b64decode(text).decode("utf-8")
        text = cleaner(text # this works because if contains html is false, then we know every part is plain
                       if not contains_html else # if contains html is true, then because of the above guardrail,
                       BeautifulSoup(text, "html.parser").get_text(strip = True, seperator = " ")) # we know its always html.
        body += text #incase there are multiple text/html parts.

    # combined text
    combined_text = f"{subject} {body}"

    # word count
    words = combined_text.split()
    word_count = len(words)

    # digit count
    digit_count = sum(c.isdigit() for c in body)

    # exclamation mark count
    exc_mark_count = body.count("!")

    # link count
    link_count = len(re.findall(r"https?://", body)) # ? next to s means s need not be present. so https and http both work fine.

    # currency count
    currency_count = (
        body.count("$") +
        body.count("₹")
    )

    # otp-like numbers
    otp_num = len(re.findall(r'\b\d{4,8}\b', body)) # \b -> word boundary, \d: check for these many digits, {4, 8} -> need to be 4-8 digits consecutively.

    
    # final record
    record = [m_id, 
              sender_name, 
              sender_local_part, 
              sender_domain, 
              combined_text, 
              word_count, 
              digit_count, 
              exc_mark_count, 
              link_count, 
              currency_count, 
              otp_num]
    
    records.append(record)

In [119]:
for message in mtype_msgs["multipart/alternative"]:
    for part in message["payload"]["parts"]:
        if part["mimeType"].startswith("image/"): print(part)

# **Okay no absolutely not surely theres a library to do all this... wait i can google this right?**
- I'm really stupid.